<a href="https://colab.research.google.com/github/LMMinier/nueronce/blob/claude/new-session-tkjr3b/cfna_colab_training_FIXED_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CFNA full training journal — repository books + multi-domain corpus

This journal trains the **35M CFNA model** from the current `LMMinier/nueronce` repository.

It dynamically discovers the books and source texts you pushed at the repository root and under:

- `books/`
- `corpus/`
- `math/`
- `psych/`
- `enlgish grammar/` and `english grammar/`

It converts `.txt`, `.tex`, `.epub`, `.html`, and `.xhtml` files into separate UTF-8 training documents, strips Project Gutenberg boilerplate, deduplicates content, creates a document-level train/validation split, and optionally adds bounded educational, math, and permissively licensed code streams.

The Drive corpus and checkpoints are keyed to the current Git commit, so the old two-record TinyStories cache cannot silently be restored.


In [ ]:
# 1) Clone/reset to the live repository branch and install dependencies.
from pathlib import Path
import json, os, shutil, subprocess, sys

REPO = "LMMinier/nueronce"
BRANCH = "claude/new-session-tkjr3b"
REPO_DIR = Path("/content/nueronce")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run([
        "git", "clone", "--branch", BRANCH, "--single-branch",
        f"https://github.com/{REPO}.git", str(REPO_DIR)
    ], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "datasets", "pytest", "cryptography", "cffi"], check=True)

os.environ["PYTHONPATH"] = str(REPO_DIR) + os.pathsep + os.environ.get("PYTHONPATH", "")
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

HEAD = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()

import torch, cfna

print("branch:", BRANCH)
print("commit:", HEAD)
print("cfna:", cfna.__file__)
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Enable a GPU runtime before training.")


In [ ]:
# 2) Safety gates before any fp16/AMP training.
os.chdir(REPO_DIR)
subprocess.run([
    sys.executable, "-m", "pytest",
    "tests/test_gpu_amp.py",
    "tests/test_prompting.py",
    "tests/test_chat_format_drift.py",
    "tests/test_mixed_sft.py",
    "-q",
], check=True)


In [ ]:
# 3) Mount Google Drive and use commit-versioned corpus/checkpoint paths.
from google.colab import drive
drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive")
CKPT_ROOT = DRIVE / "cfna_checkpoints"
CKPT_ROOT.mkdir(parents=True, exist_ok=True)

VERSION = f"repo_books_{HEAD[:12]}"
LOCAL_CORPUS = Path(f"/content/cfna_corpus_{VERSION}")
DRIVE_CORPUS = DRIVE / f"cfna_corpus_{VERSION}"
BASE_CKPT = CKPT_ROOT / f"cfna_base_35m_{VERSION}.pt"
SFT_DATA = Path("/content/cfna_conversation_sft")
SFT_DIR = CKPT_ROOT / f"cfna_conversation_35m_{VERSION}"

print("version:", VERSION)
print("Drive corpus:", DRIVE_CORPUS)
print("base checkpoint:", BASE_CKPT)
print("SFT checkpoint directory:", SFT_DIR)


In [ ]:
# 4) Inspect the exact newly pushed repository files that will be converted.
from collections import Counter

SUFFIXES = {".txt", ".tex", ".epub", ".html", ".htm", ".xhtml"}
SCAN_DIRS = ("books", "corpus", "math", "psych", "enlgish grammar", "english grammar")

files = {
    p.resolve()
    for p in REPO_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in SUFFIXES
}
for name in SCAN_DIRS:
    root = REPO_DIR / name
    if root.is_dir():
        files |= {
            p.resolve()
            for p in root.rglob("*")
            if p.is_file() and p.suffix.lower() in SUFFIXES
        }

files = sorted(files, key=lambda p: str(p.relative_to(REPO_DIR)).lower())
if not files:
    raise RuntimeError("No repository books/source files were discovered.")

groups = Counter(
    p.relative_to(REPO_DIR).parts[0] if len(p.relative_to(REPO_DIR).parts) > 1 else "<repo-root>"
    for p in files
)

print(f"discovered {len(files)} repository source files")
for group, count in groups.most_common():
    print(f"{group:25s} {count:4d}")

print("\nSample files:")
for p in files[:30]:
    print(" -", p.relative_to(REPO_DIR))


In [ ]:
# 5) Build or restore the exact commit-versioned corpus.
#
# Repository books are always included. The bounded HF additions provide
# educational prose, math, and permissively licensed code without letting one
# giant source replace the books.
if DRIVE_CORPUS.exists() and (DRIVE_CORPUS / "manifest.jsonl").exists():
    if LOCAL_CORPUS.exists():
        shutil.rmtree(LOCAL_CORPUS)
    shutil.copytree(DRIVE_CORPUS, LOCAL_CORPUS)
    print("restored this commit's corpus from Drive")
else:
    if LOCAL_CORPUS.exists():
        shutil.rmtree(LOCAL_CORPUS)

    build_cmd = [
        sys.executable, "-u", str(REPO_DIR / "scripts/build_full_corpus.py"),
        "--repo-root", str(REPO_DIR),
        "--out", str(LOCAL_CORPUS),
        "--include-hf",
        "--hf-job", "tinystories=40000000",
        "--hf-job", "cosmopedia_100k=80000000",
        "--hf-job", "fineweb_edu_sample_10bt=60000000",
        "--hf-job", "open_web_math=50000000",
        "--hf-job", "the_stack_smol=40000000",
        "--hf-shard-bytes", "4000000",
        "--hf-val-every", "10",
        "--val-fraction", "0.10",
        "--min-bytes", "2000",
        "--seed", "42",
    ]
    print("launching corpus build:")
    print(" ".join(build_cmd))
    subprocess.run(build_cmd, cwd=REPO_DIR, env=os.environ.copy(), check=True)

    drive_tmp = DRIVE_CORPUS.with_name(DRIVE_CORPUS.name + ".incomplete")
    if drive_tmp.exists():
        shutil.rmtree(drive_tmp)
    shutil.copytree(LOCAL_CORPUS, drive_tmp)
    drive_tmp.rename(DRIVE_CORPUS)
    print("saved completed corpus to Drive")

CORPUS_DIR = str(LOCAL_CORPUS)


In [ ]:
# 6) Hard corpus audit. This refuses the earlier two-record TinyStories-only failure.
from collections import Counter, defaultdict

manifest_path = LOCAL_CORPUS / "manifest.jsonl"
records = [
    json.loads(line)
    for line in manifest_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

missing = [r["path"] for r in records if not (LOCAL_CORPUS / r["path"]).is_file()]
empty = [r["path"] for r in records
         if (LOCAL_CORPUS / r["path"]).is_file()
         and (LOCAL_CORPUS / r["path"]).stat().st_size == 0]

splits = Counter(r["split"] for r in records)
source_docs = Counter(r["source_collection"] for r in records)
source_bytes = defaultdict(int)
repo_origins = Counter()

for r in records:
    source_bytes[r["source_collection"]] += int(r.get("n_bytes", 0))
    locator = str(r.get("source_locator", ""))
    if locator.startswith("repo://"):
        rel = locator[len("repo://"):]
        repo_origins[rel.split("/", 1)[0] if "/" in rel else "<repo-root>"] += 1

total_bytes = sum(source_bytes.values())
repo_docs = sum(1 for r in records if str(r.get("source_locator", "")).startswith("repo://"))

print("manifest records:", len(records))
print("train/val:", dict(splits))
print("repository documents:", repo_docs)
print("total size:", f"{total_bytes / 1e6:.2f} MB")
print("repository origins:", dict(repo_origins))

print("\nSources:")
for name, count in source_docs.most_common():
    print(f"{name:48s} {count:4d} files  {source_bytes[name]/1e6:8.2f} MB")

assert not missing, f"missing corpus files: {missing[:10]}"
assert not empty, f"empty corpus files: {empty[:10]}"
assert len(records) >= 20, "too few manifest documents"
assert repo_docs >= 10, "the pushed repository books were not included"
assert splits["train"] > 0 and splits["val"] >= 2, "invalid document-level split"
assert total_bytes >= 20_000_000, "corpus is unexpectedly small"
assert not (len(records) == 2 and len(source_docs) == 1), "old two-record corpus detected"

print("\nCORPUS AUDIT PASSED")


In [ ]:
# 7) Base pretrain: 34.4M parameters, atomic checkpoint saves, CUDA-safe resume.
from datetime import datetime

def checkpoint_is_valid(path: Path) -> bool:
    if not path.exists():
        return False
    try:
        ck = torch.load(path, map_location="cpu", weights_only=False)
        return "state_dict" in ck and "config" in ck
    except Exception as exc:
        bad = path.with_name(
            path.name + ".corrupt_" + datetime.now().strftime("%Y%m%d_%H%M%S")
        )
        path.rename(bad)
        print("preserved corrupt checkpoint:", bad)
        print(repr(exc))
        return False

base_cmd = [
    sys.executable, "-u", str(REPO_DIR / "scripts/train_checkpoint.py"),
    "--preset", "base_35m",
    "--corpus", CORPUS_DIR,
    "--minutes", "170",
    "--seq", "192",
    "--batch", "16",
    "--lr", "3e-4",
    "--amp",
    "--device", "cuda",
    "--out", str(BASE_CKPT),
]
if checkpoint_is_valid(BASE_CKPT):
    base_cmd.append("--resume")

print("launching base training:")
print(" ".join(base_cmd))
subprocess.run(base_cmd, cwd=REPO_DIR, env=os.environ.copy(), check=True)


In [ ]:
# 8) Report convergence and enforce the base-quality gate before SFT.
ck = torch.load(BASE_CKPT, map_location="cpu", weights_only=False)
history = ck.get("history", [])
if not history:
    raise RuntimeError("The base checkpoint has no training history.")

best = min(history, key=lambda row: row["heldout_bpb"])
last = history[-1]
tail = history[-10:]
tail_gain = (
    tail[0]["heldout_bpb"] - tail[-1]["heldout_bpb"]
    if len(tail) > 1 else 0.0
)

print("checkpoint step:", ck.get("step"))
print("last held-out bpb:", round(last["heldout_bpb"], 4))
print("best held-out bpb:", round(best["heldout_bpb"], 4), "at step", best["step"])
print("last-10-evaluation improvement:", round(tail_gain, 4))

BASE_GATE = best["heldout_bpb"] < 1.8
TARGET_GATE = best["heldout_bpb"] <= 1.5
print("SFT gate < 1.8:", "PASS" if BASE_GATE else "FAIL")
print("35M target <= 1.5:", "PASS" if TARGET_GATE else "NOT YET")

if not BASE_GATE:
    raise RuntimeError(
        "Base held-out bpb is not below 1.8 yet. Rerun Cell 7 to resume base training "
        "before starting conversational SFT."
    )


In [ ]:
# 9) Build the deterministic 54K conversational SFT dataset.
if SFT_DATA.exists():
    shutil.rmtree(SFT_DATA)

sft_build_cmd = [
    sys.executable, "-u", str(REPO_DIR / "scripts/build_conversation_sft.py"),
    "--out-dir", str(SFT_DATA),
]
print(" ".join(sft_build_cmd))
subprocess.run(sft_build_cmd, cwd=REPO_DIR, env=os.environ.copy(), check=True)

manifest = SFT_DATA / "manifest.json"
if manifest.exists():
    print(manifest.read_text(encoding="utf-8")[:5000])
else:
    print("SFT dataset built:", SFT_DATA)


In [ ]:
# 10) Response-masked conversational SFT, warm-started from the trained 35M base.
#
# Re-running this cell resumes from latest.pt. best.pt is selected by validation,
# not merely the final step.
sft_cmd = [
    sys.executable, "-u", str(REPO_DIR / "scripts/train_conversation.py"),
    "--data", str(SFT_DATA),
    "--preset", "base_35m",
    "--out-dir", str(SFT_DIR),
    "--loss", "response",
    "--minutes", "120",
    "--batch", "8",
    "--max-len", "320",
    "--lr", "1e-4",
    "--val-every", "200",
    "--val-batches", "16",
    "--amp",
    "--device", "cuda",
    "--metrics-out", str(SFT_DIR / "conversation_train.jsonl"),
]

latest = SFT_DIR / "latest.pt"
if latest.exists() and checkpoint_is_valid(latest):
    sft_cmd.append("--resume")
else:
    sft_cmd += ["--init-from", str(BASE_CKPT)]

print("launching conversational SFT:")
print(" ".join(sft_cmd))
subprocess.run(sft_cmd, cwd=REPO_DIR, env=os.environ.copy(), check=True)


In [ ]:
# 11) Evaluate the best conversational checkpoint with real generated replies.
BEST_SFT = SFT_DIR / "best.pt"
if not BEST_SFT.exists():
    BEST_SFT = SFT_DIR / "latest.pt"

chat_cmd = [
    sys.executable, "-u", str(REPO_DIR / "scripts/chat_demo.py"),
    "--ckpt", str(BEST_SFT),
    "--temp", "0.2",
    "--max-new", "120",
    "--seed", "0",
]
print(" ".join(chat_cmd))
subprocess.run(chat_cmd, cwd=REPO_DIR, env=os.environ.copy(), check=True)


In [ ]:
# 12) Optional interactive chat.
from cfna.chat import Conversation, load_checkpoint

model, final_ckpt = load_checkpoint(str(BEST_SFT))
chat = Conversation(
    model,
    system="You are CFNA, a careful and direct assistant.",
    temperature=0.2,
    max_new=160,
)

print("Type quit to stop.")
while True:
    user_text = input("You: ").strip()
    if user_text.lower() in {"quit", "exit"}:
        break
    print("CFNA:", chat.say(user_text))
